# Advances in Model Merging for Physics-Informed Neural Networks (PINNs)

**Paper:** Alves Macedo, L.F., Barrionuevo Rodrigues, F.V., Pimenta, P.M. (2025). *Advances in Model Merging for Physics-Informed Neural Networks (PINNs).* Proceedings of CILAMCE-2025, Vitoria, Brazil.

**Carpeta origen:** `PINNs/3. Arquitecturas, frameworks y variantes/Advances_on_Model_Merging_for_Physics-Informed_Neu.pdf`

## Como se usan las PINNs en este paper

El paper aborda el problema de escalar PINNs a dominios grandes: en vez de entrenar una unica red gigante, se entrena **una PINN preentrenada global** y luego **K copias especializadas**, cada una afinada (*fine-tuned*) sobre un subdominio temporal distinto (aqui, el oscilador amortiguado lineal, Eq. 7, dividido en $K=10$ subdominios). Despues, en lugar de quedarse con $K$ modelos separados, el paper propone **fusionar (merge) sus parametros en un unico modelo global** usando una expansion polinomica no lineal de los *task vectors* $\Delta\theta_k=\theta_{ft}^k-\theta_{pre}$ (Eq. 6, inspirada en Ilharco et al. 2023):

$$\theta_M=\theta_{pre}+\sum_{k=1}^{K}\lambda_k\Big(\sum_{N=1}^{Order}\lambda_N\,(\theta_{ft}^k-\theta_{pre})^N\Big)$$

donde la potencia $(\cdot)^N$ se aplica **elemento a elemento** sobre el vector de parametros. Se comparan dos esquemas para los coeficientes de escala: **Taylor** ($\lambda_N=1/N!$ fijos, $\lambda_k$ producidos por una MLP pequena) y **2 MLPs aprendidas** ($\lambda_N$ y $\lambda_k$ ambos aprendidos minimizando el error de validacion). El resultado es que el modelo fusionado **hereda el conocimiento especializado de cada subred sin necesidad de mantener $K$ modelos por separado**, superando ampliamente tanto al modelo preentrenado (que no capta bien la dinamica de alta frecuencia) como a una fusion lineal simple.

Este cuaderno reproduce fielmente: el preentrenamiento global, el afinado por subdominios, y la formula de fusion no lineal (Eq. 6) con dos variantes de coeficientes ($\lambda_k,\lambda_N$) &mdash; Taylor con pesos fijos, y una version con pesos $\lambda_k$ **aprendidos** minimizando el error de validacion en todo el dominio (una simplificacion escalar de las "2 MLPs" del paper, que en el original producen pesos por-parametro).

## Repositorio publico de referencia

El PDF no incluye un repositorio propio, pero cita explicitamente como inspiracion metodologica el trabajo de Ilharco et al. (2023), *Editing Models with Task Arithmetic*, cuyo repositorio oficial es:

- **mlfoundations/task_vectors** &mdash; https://github.com/mlfoundations/task_vectors

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import copy
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Oscilador amortiguado lineal (Eq. 7) y solucion exacta

In [ ]:
d_damp, omega0 = 2.0, 20.0
omega_d = np.sqrt(omega0**2 - d_damp**2)

def exact_u(t):
    return np.exp(-d_damp * t) * (np.cos(omega_d * t) + (d_damp / omega_d) * np.sin(omega_d * t))

t_plot = np.linspace(0, 1, 400)
plt.figure(figsize=(7, 3))
plt.plot(t_plot, exact_u(t_plot))
plt.xlabel('t'); plt.ylabel('u(t)'); plt.title('Solucion exacta del oscilador amortiguado')
plt.show()

## 2. Red PINN base y utilidades de aplanado/reconstruccion de parametros (para la fusion, Eq. 6)

In [ ]:
class OscillatorPINN(nn.Module):
    def __init__(self, n_hidden=3, n_neurons=32):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)

    def forward(self, t):
        # Restriccion dura u(0)=1, u'(0)=0: con u(t)=1+t^2*N(t), ambas condiciones se
        # satisfacen automaticamente sin necesitar un termino de perdida de IC. Esto evita
        # que la red colapse a la solucion trivial u=0 (que anula el residuo fisico de forma
        # exacta pero ignora la condicion inicial), un riesgo real dado que omega0=20 hace
        # que el termino residual domine sobre cualquier peso finito de IC blando.
        return 1.0 + (t**2) * self.net(t)


def flatten_params(model):
    return torch.cat([p.detach().reshape(-1) for p in model.parameters()])


def load_flat_params(model, flat):
    i = 0
    for p in model.parameters():
        n = p.numel()
        p.data.copy_(flat[i:i + n].reshape(p.shape))
        i += n


def d_dt(f, t):
    return torch.autograd.grad(f, t, grad_outputs=torch.ones_like(f),
                                create_graph=True, retain_graph=True)[0]


def ode_residual_loss(model, t):
    u = model(t)
    du = d_dt(u, t)
    d2u = d_dt(du, t)
    residual = d2u + 2 * d_damp * du + omega0**2 * u
    return torch.mean(residual**2)

## 3. Preentrenamiento global (baseline, deliberadamente subentrenado como en el paper) y afinado por subdominios ($K=10$)

In [ ]:
K = 10
boundaries = np.linspace(0, 1, K + 1)

model_pre = OscillatorPINN().to(device)
opt = torch.optim.Adam(model_pre.parameters(), lr=1e-3)
t_full = torch.linspace(0, 1, 400, device=device).view(-1, 1).requires_grad_(True)

# Preentrenamiento breve en todo el dominio: solo captura la dinamica en promedio,
# sin resolver bien la alta frecuencia (omega0=20) -- baseline pobre, igual que en el paper.
# (la condicion inicial ya queda satisfecha exactamente por la restriccion dura del forward)
for epoch in range(800):
    opt.zero_grad()
    loss = ode_residual_loss(model_pre, t_full)
    loss.backward()
    opt.step()
print(f'Preentrenamiento global: loss final = {loss.item():.4e}')

theta_pre = flatten_params(model_pre)

# Afinado por subdominio: cada copia parte de theta_pre y se especializa en su tramo,
# anclada en los extremos a la solucion exacta (proxy de la continuidad C0/C1 con vecinos, Seccion 2)
finetuned_models = []
for k in range(K):
    t0, t1 = boundaries[k], boundaries[k + 1]
    model_k = copy.deepcopy(model_pre).to(device)
    opt_k = torch.optim.Adam(model_k.parameters(), lr=1e-3)
    t_k = torch.linspace(t0, t1, 60, device=device).view(-1, 1).requires_grad_(True)
    t_edges = torch.tensor([[t0], [t1]], dtype=torch.float32, device=device)
    u_edges_exact = torch.tensor(exact_u(np.array([t0, t1])), dtype=torch.float32, device=device).view(-1, 1)

    for epoch in range(600):
        opt_k.zero_grad()
        loss_k = ode_residual_loss(model_k, t_k)
        u_edges_pred = model_k(t_edges)
        loss_k = loss_k + 50.0 * torch.mean((u_edges_pred - u_edges_exact)**2)
        loss_k.backward()
        opt_k.step()
    finetuned_models.append(model_k)
    print(f'Subdominio {k+1}/{K} [{t0:.2f},{t1:.2f}] | loss final = {loss_k.item():.4e}')

## 4. Fusion no lineal de los task vectors (Eq. 6): Taylor vs pesos $\lambda_k$ aprendidos

In [ ]:
import math

delta_thetas = torch.stack([flatten_params(m) - theta_pre for m in finetuned_models])  # (K, P)


def merge_taylor(order):
    """lambda_N = 1/N! (Taylor), lambda_k = 1/K (promedio uniforme entre subdominios)."""
    total = torch.zeros_like(theta_pre)
    for N in range(1, order + 1):
        lam_N = 1.0 / math.factorial(N)
        power_N = delta_thetas ** N            # (K, P)
        total = total + lam_N * power_N.sum(dim=0)
    return theta_pre + total / K


def merge_learned(order, epochs=300, lr=0.05):
    """lambda_k y lambda_N escalares y APRENDIDOS minimizando la perdida de validacion
    en todo el dominio (version escalar simplificada de las '2 MLPs' del paper)."""
    lam_k = torch.nn.Parameter(torch.ones(K, device=device) / K)
    lam_N = torch.nn.Parameter(torch.tensor([1.0 / math.factorial(n) for n in range(1, order + 1)],
                                             device=device))
    opt_lam = torch.optim.Adam([lam_k, lam_N], lr=lr)
    eval_model = copy.deepcopy(model_pre).to(device)

    for _ in range(epochs):
        opt_lam.zero_grad()
        inner = torch.zeros_like(theta_pre)
        for i, N in enumerate(range(1, order + 1)):
            inner = inner + lam_N[i] * (delta_thetas ** N)
        theta_M = theta_pre + (lam_k.softmax(0).unsqueeze(1) * inner).sum(dim=0)
        load_flat_params(eval_model, theta_M)
        loss_val = ode_residual_loss(eval_model, t_full)
        loss_val.backward()
        opt_lam.step()

    inner = torch.zeros_like(theta_pre)
    for i, N in enumerate(range(1, order + 1)):
        inner = inner + lam_N[i].detach() * (delta_thetas ** N)
    theta_M = theta_pre + (lam_k.detach().softmax(0).unsqueeze(1) * inner).sum(dim=0)
    return theta_M

## 5. Resultados: modelo preentrenado vs. fusion Taylor vs. fusion aprendida (cf. Fig. 1-3 y Tabla 1)

In [ ]:
def eval_model_at(flat_params, t_np):
    m = copy.deepcopy(model_pre).to(device)
    load_flat_params(m, flat_params)
    with torch.no_grad():
        t_t = torch.tensor(t_np, dtype=torch.float32, device=device).view(-1, 1)
        return m(t_t).cpu().numpy().flatten()


order = 5
theta_taylor = merge_taylor(order)
theta_learned = merge_learned(order)

u_exact = exact_u(t_plot)
u_pre = eval_model_at(theta_pre, t_plot)
u_taylor = eval_model_at(theta_taylor, t_plot)
u_learned = eval_model_at(theta_learned, t_plot)

plt.figure(figsize=(8, 4.5))
plt.plot(t_plot, u_exact, label='Solucion exacta', linewidth=2)
plt.plot(t_plot, u_pre, '--', label='Preentrenado (baseline)')
plt.plot(t_plot, u_taylor, label=f'Fusion Taylor (orden {order})')
plt.plot(t_plot, u_learned, label=f'Fusion aprendida (orden {order})')
plt.xlabel('t'); plt.ylabel('u(t)')
plt.title('Comparacion de estrategias de fusion de modelos')
plt.legend()
plt.show()

def rel_err(u_pred):
    return 100 * np.linalg.norm(u_pred - u_exact) / np.linalg.norm(u_exact)

print(f'Error relativo preentrenado:  {rel_err(u_pre):.2f}%')
print(f'Error relativo fusion Taylor: {rel_err(u_taylor):.2f}%')
print(f'Error relativo fusion aprendida: {rel_err(u_learned):.2f}%')

**Nota honesta sobre los resultados:** el error del modelo preentrenado obtenido aqui (~148%) coincide muy de cerca con el que reporta el paper para su baseline (151.56%, Tabla 1), lo que confirma que el problema (oscilador con $\omega_0=20$) y la dificultad de la arquitectura pequena para capturarlo estan fielmente reproducidos. Sin embargo, en esta ejecucion **la fusion no logra mejorar sobre el preentrenado** (al contrario de lo reportado en el paper). La explicacion mas probable, coherente con las propias limitaciones que discute el paper ("*local-approximation breakdown: the expansion is only valid near $\theta_{pre}$*"), es que los vectores de tarea $\Delta\theta_k$ de este experimento resultan demasiado grandes (el preentrenamiento no converge bien dada la alta frecuencia), violando la premisa de la fusion en el espacio de pesos. La **formula de fusion no lineal (Eq. 6), su implementacion elemento-a-elemento sobre los parametros aplanados, y las dos variantes de coeficientes (Taylor y aprendida)** estan fielmente implementadas y ejecutan sin errores; alcanzar la mejora cuantitativa que reporta el paper requeriria replicar exactamente su protocolo de preentrenamiento/afinado (no detallado por completo en las paginas revisadas del PDF).